In [1]:
#!/usr/bin/env python
# coding: utf-8

import os
import anndata as ad
import numpy as np
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# import omicverse as ov # Commented out as it's not used in this snippet
import scvi
from scvi.model.utils import mde
from scarches.models.scpoli import scPoli
from scarches.dataset.trvae.data_handling import remove_sparsity

import warnings
warnings.filterwarnings('ignore')

# Note: IPython magic commands are typically not used in script files.
# If running as a script, these lines can be removed or commented out.
# %load_ext autoreload
# %autoreload 2

sc.settings.set_figure_params(dpi=100, frameon=False)
sc.set_figure_params(dpi=100)
sc.set_figure_params(figsize=(3, 3))
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (3, 3)


# --- Configuration ---
# Change the working directory to the Garfield folder (if needed)
# os.chdir('/storage2/liuxiaodongLab/fanxueying/embryo_benchmarking_rebuttal/code/20250801_training_model_evaluation_batch')
# print(os.getcwd())
os.chdir('/storage2/liuxiaodongLab/fanxueying/embryo_benchmarking_rebuttal/code/20250801_training_model_evaluation_batch')
os.getcwd()

# List of h5ad files to process
h5ad_files_to_process = [
   # "/storage2/liuxiaodongLab/fanxueying/embryo_benchmarking_rebuttal/code/20250729_scpoli_optimization_v3/embryo_model_integration_scPoli.h5ad",
    # "/storage2/liuxiaodongLab/fanxueying/embryo_benchmarking_rebuttal/code/20250729_scpoli_optimization_v3/embryo_model_integration_scPoli_2round.h5ad",
    #"/storage2/liuxiaodongLab/fanxueying/embryo_benchmarking_rebuttal/code/20250729_scpoli_optimization_v3/embryo_model_integration_scPoli_3round.h5ad",
   # "/storage2/liuxiaodongLab/fanxueying/embryo_benchmarking_rebuttal/code/20250731_scpoli_optimization_comparasion_v3/embryo_model_integration_scPoli_balance_data.h5ad",
    "/storage2/liuxiaodongLab/fanxueying/embryo_benchmarking_rebuttal/code/20250731_scpoli_optimization_comparasion_v3/embryo_model_integration_scPoli_balance_weight.h5ad"
]

# Output directory for plots and modified h5ad files
output_dir = './processed_models_and_plots' # Update this path as needed
figures_folder = os.path.join(output_dir, 'umap_plots')
os.makedirs(figures_folder, exist_ok=True)
modified_h5ad_folder = os.path.join(output_dir, 'modified_h5ad_files')
os.makedirs(modified_h5ad_folder, exist_ok=True)

# Your ordered labels list for annotation visualization
ordered_labels = [
    'TE', 'CTB_1','CTB_2', 'STB_1', 'STB_2', 'STB_3', 'EVT_1', 'EVT_2',
    'Epiblast_1','Epiblast_2','Epiblast_3','Ectoderm',
    'Amniontic.epi','Amniontic.ectoderm',
    'PGC',
    'Primitive.streak',
    'Neuromesodermal.progenitor',
    'Neural.crest', 'Neural.ectoderm.forebrain', 'Neural.ectoderm.hindbrain', 'Neural.ectoderm.midbrain','Spinal.cord',
    'Paraxial.mesoderm','Emergent.mesoderm','Pre-somatic.mesoderm','Somite', 'Rostral.mesoderm', 'Lateral.plate.mesoderm_1',
    'Lateral.plate.mesoderm_2','Lateral.plate.mesoderm_3','Cardiac.mesoderm','Amniotic.mesoderm','Exe.meso.progenitor','YS.mesoderm_1', 'YS.mesoderm_2',
    'Hypoblast_1', 'Hypoblast_2', 'AVE', 'VE', 'YS.endoderm',
    'DE','Gut',
    'Notochord',
    'Hemogenic.endothelial.progenitor','Endothelium','Erythroid','Primitive.megakaryocyte','Myeloid.progenitor'
]

# --- Process each h5ad file ---
for i, h5ad_file_path in enumerate(h5ad_files_to_process):
    print(f"\n{'='*50}")
    print(f"Processing file {i+1}/{len(h5ad_files_to_process)}: {h5ad_file_path}")
    print(f"{'='*50}")

    if not os.path.exists(h5ad_file_path):
        print(f"Warning: File not found, skipping: {h5ad_file_path}")
        continue

    try:
        # Load the annotated dataset
        adata = sc.read_h5ad(h5ad_file_path)
        file_name = os.path.splitext(os.path.basename(h5ad_file_path))[0]
        print(f"Loaded data '{file_name}' with shape: {adata.shape}")

        # List available columns
        print(f"Available columns in obs: {list(adata.obs.columns)}")

        # --- Process reanno_pred ---
        reanno_pred_column = 'human_ref_reanno_pred' # Assuming the column name is 'reanno_pred'
        if reanno_pred_column in adata.obs.columns:
            print(f"Processing '{reanno_pred_column}' column...")
            unique_annos_in_data = adata.obs[reanno_pred_column].unique()
            print(f"Unique annotation values in data: {len(unique_annos_in_data)} found")
            print(f"Sample unique values: {list(unique_annos_in_data)[:10]}...") # Print first 10

            # --- Key Change: Set categories to the full ordered list ---
            # This ensures the order is preserved, even for missing categories
            # Cells with annotations not in ordered_labels will become NaN in the categorical
            # You might want to handle these NaNs, e.g., by filtering or adding an 'Unknown' category
            # For now, we'll keep them as NaN, which scanpy usually handles by not plotting them or coloring them grey.

            # Ensure all values in the data are strings (good practice)
            adata.obs[reanno_pred_column] = adata.obs[reanno_pred_column].astype(str)

            # Create Categorical with the full ordered list as categories
            # Categories not present in the data will be unused levels
            adata.obs[reanno_pred_column] = pd.Categorical(
                adata.obs[reanno_pred_column],
                categories=ordered_labels, # Use the full, ordered list
                ordered=True
            )
            print(f"'{reanno_pred_column}' column converted to ordered categorical with {len(ordered_labels)} levels.")

            # Check for any values in data that are NOT in ordered_labels (these will become NaN)
            invalid_categories = set(unique_annos_in_data) - set(ordered_labels)
            if invalid_categories:
                print(f"Warning: Found categories in data not in ordered_labels (will be set to NaN): {invalid_categories}")
                # Optionally filter out these cells or add them to ordered_labels
                # For now, we leave them as NaN in the categorical


            # --- Compute UMAP (if needed) ---
            # Check if UMAP embedding exists
            if 'X_umap' not in adata.obsm:
                print("Computing UMAP coordinates as 'X_umap' not found...")
                # Basic preprocessing if needed (check if raw counts are in .X or layers)
                # This is a simplified check, adapt based on your data's state
                if not (adata.X.max() < 50 and adata.X.min() >= 0): # Heuristic: if values look like counts
                     print("Data appears to be raw counts, normalizing and log-transforming...")
                     sc.pp.normalize_total(adata, target_sum=1e4)
                     sc.pp.log1p(adata)
                else:
                     print("Data appears to be normalized/logged.")

                # Find HVGs if not already done
                if 'highly_variable' not in adata.var.columns:
                    print("Finding highly variable genes...")
                    try:
                        sc.pp.highly_variable_genes(adata, n_top_genes=min(2000, adata.n_vars - 1), flavor='seurat')
                    except:
                         print("Finding HVGs failed, using all genes.")
                         adata.var['highly_variable'] = True

                # Compute PCA
                print("Computing PCA...")
                sc.tl.pca(adata, svd_solver='arpack', use_highly_variable=True if 'highly_variable' in adata.var.columns else False)

                # Compute Neighbors
                print("Computing Neighbors...")
                sc.pp.neighbors(adata, n_neighbors=10, n_pcs=30) # Adjust parameters if needed

                # Compute UMAP
                print("Computing UMAP...")
                sc.tl.umap(adata)
                print("UMAP computation completed.")
            else:
                print("Using existing UMAP coordinates from 'X_umap'.")


            # --- Plot UMAP with annotation colors ---
            print("Plotting UMAP with cell type annotations...")
            plt.figure(figsize=(14, 10)) # Adjust size for potentially many categories

            # Use scanpy's default coloring, which respects the categorical order
            # If you have specific colors, define an `anno_color` dict and use `palette=anno_color`
            sc.pl.umap(
                adata,
                color=reanno_pred_column,
                # palette=anno_color, # Uncomment and define anno_color if you have specific colors
                title=f"{file_name} - Cell Type Annotation (Ordered)",
                show=False, # Don't show, we'll save manually
                frameon=False
            )

            # Save the plot
            plot_file_name = f"{file_name}_reanno_pred_umap.pdf"
            plot_outfile = os.path.join(figures_folder, plot_file_name)
            plt.savefig(plot_outfile, dpi=300, bbox_inches='tight')
            plt.close()
            print(f"Saved annotation UMAP plot to {plot_outfile}")

            # --- Save the modified h5ad file ---
            print("Saving modified AnnData object...")
            modified_h5ad_file_name = f"{file_name}_with_ordered_reanno.h5ad"
            modified_h5ad_outfile = os.path.join(modified_h5ad_folder, modified_h5ad_file_name)

            # Ensure sparse matrices are handled if needed (optional, but good practice)
            # adata = remove_sparsity(adata)
            # print("obsm keys before saving:", list(adata.obsm.keys()))

            adata.write_h5ad(modified_h5ad_outfile)
            print(f"Saved modified h5ad file to {modified_h5ad_outfile}")

        else:
            print(f"Warning: '{reanno_pred_column}' column not found in the dataset '{file_name}'. Skipping plotting and saving for this file.")

        print(f"Finished processing {file_name}.\n")


    except Exception as e:
        print(f"Error processing file {h5ad_file_path}: {e}")
        import traceback
        traceback.print_exc()
        continue # Continue with the next file

print(f"\n{'='*50}")
print("All files processed!")
print(f"Plots saved to: {figures_folder}")
print(f"Modified h5ad files saved to: {modified_h5ad_folder}")
print(f"{'='*50}")


[rank: 0] Global seed set to 0
/home/liuxiaodongLab/fanxueying/miniconda3/envs/benchmarking/lib/python3.8/site-packages/flax/struct.py:132: FutureWarning: jax.tree_util.register_keypaths is deprecated, and will be removed in a future release. Please use `register_pytree_with_keys()` instead.
  jax.tree_util.register_keypaths(data_clz, keypaths)
/home/liuxiaodongLab/fanxueying/miniconda3/envs/benchmarking/lib/python3.8/site-packages/flax/struct.py:132: FutureWarning: jax.tree_util.register_keypaths is deprecated, and will be removed in a future release. Please use `register_pytree_with_keys()` instead.
  jax.tree_util.register_keypaths(data_clz, keypaths)



Processing file 1/1: /storage2/liuxiaodongLab/fanxueying/embryo_benchmarking_rebuttal/code/20250731_scpoli_optimization_comparasion_v3/embryo_model_integration_scPoli_balance_weight.h5ad
Loaded data 'embryo_model_integration_scPoli_balance_weight' with shape: (137647, 35142)
Available columns in obs: ['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'sample_type', 'scmap_nakamura', 'scmapCELL_Yang', 'scmap_ma', 'scmap_Tyser', 'scmapCELL_Mole', 'cell_assignment', 'course_cell_assignment', 'stage', 'species', 'embryo', 'platform', 'doublet', 'doublet_score', 'human_ref_lineage_pred', 'human_ref_lineage_uncert', 'human_ref_reanno_pred', 'human_ref_reanno_uncert', 'batch', 'Unintegrated_res_0.5', 'scANVI_res_0.5']
Processing 'human_ref_reanno_pred' column...
Unique annotation values in data: 48 found
Sample unique values: ['Lateral.plate.mesoderm_1', 'Amniontic.ectoderm', 'Amniotic.mesoderm', 'YS.mesoderm_2', 'Exe.meso.progenitor', 'Lateral.plate.mesoderm_3', 'Gut', 'Amniontic.ep

<Figure size 1400x1000 with 0 Axes>

In [2]:
adata

AnnData object with n_obs × n_vars = 137647 × 35142
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'sample_type', 'scmap_nakamura', 'scmapCELL_Yang', 'scmap_ma', 'scmap_Tyser', 'scmapCELL_Mole', 'cell_assignment', 'course_cell_assignment', 'stage', 'species', 'embryo', 'platform', 'doublet', 'doublet_score', 'human_ref_lineage_pred', 'human_ref_lineage_uncert', 'human_ref_reanno_pred', 'human_ref_reanno_uncert', 'batch', 'Unintegrated_res_0.5', 'scANVI_res_0.5'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'

In [1]:
import os
import anndata as ad
import numpy as np
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# import omicverse as ov # Commented out as it's not used in this snippet

import warnings
warnings.filterwarnings('ignore')

adata = sc.read_h5ad("/storage2/liuxiaodongLab/fanxueying/embryo_benchmarking_rebuttal/code/20250801_training_model_evaluation_batch/processed_models_and_plots/modified_h5ad_files/embryo_model_integration_scPoli_3round_with_ordered_reanno.h5ad")
adata

AnnData object with n_obs × n_vars = 137647 × 35142
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'sample_type', 'scmap_nakamura', 'scmapCELL_Yang', 'scmap_ma', 'scmap_Tyser', 'scmapCELL_Mole', 'cell_assignment', 'course_cell_assignment', 'stage', 'species', 'embryo', 'platform', 'doublet', 'doublet_score', 'human_ref_lineage_pred', 'human_ref_lineage_uncert', 'human_ref_reanno_pred', 'human_ref_reanno_uncert', 'batch', 'Unintegrated_res_0.5', 'scANVI_res_0.5'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: 'human_ref_lineage_pred_colors', 'human_ref_reanno_pred_colors', 'hvg', 'leiden', 'log1p', 'neighbors', 'orig.ident_colors', 'pca', 'stage_colors', 'umap'
    obsm: 'Unintegrated', 'X_Unintegrated', 'X_pca', 'X_scANVI', 'X_umap', 'scANVI', 'scVI'
    varm: 'PCs'
    layers: 'counts', 'logcounts'
    obsp: 'connectivities', 'distances'

In [ ]:
sc.pl.umap(
    adata,
    color="human_ref_lineage_uncert",
    show=True,
    frameon=False,
    cmap='magma',
    vmax=1,
    save='embryo_model_integration_scPoli_3round_lineage_uncert.pdf'
)
    

In [ ]:
sc.pl.umap(
    adata,
    color="human_ref_reanno_uncert",
    show=True,
    frameon=False,
    cmap='magma',
    vmax=1,
    save='embryo_model_integration_scPoli_3round_reanno_uncert.pdf'
)

In [13]:
import os
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt

sc.settings.set_figure_params(dpi=100, frameon=False)
sc.set_figure_params(dpi=100)
sc.set_figure_params(figsize=(3, 3))
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (3, 3)

# Define list of genes
genes_to_plot = [
    "POU5F1", "NANOG",  # epiblast
    "SOX2", "TTYH1",    # neural ectoderm
    "GATA3", "TFAP2A",
    "TBXT", "CDX1", "PDGFRA", "APOA2", "FOXA2", "NANOS3", "TAL1", "ISL1",
    "COL6A1", "COL6A2", "GABRP", "HEY1", "PTPRC", "HBM", "PECAM1", "HBZ", "DLX5",
]

# Create output directory (if it doesn't exist)
output_dir = "figures/gene_expression_plots"
os.makedirs(output_dir, exist_ok=True)

# Loop over each gene and plot/save individually
for gene in genes_to_plot:
    fig, ax = plt.subplots(figsize=(6, 5))  # Create a new figure for each gene
    sc.pl.umap(
        adata,
        color=gene,
        use_raw=False,
        cmap=sns.cubehelix_palette(dark=0, light=.9, as_cmap=True),
        ax=ax,
        show=False,  # Prevents Scanpy from calling plt.show()
        size=15
    )
    plt.tight_layout()
    plt.savefig(f"{output_dir}/{gene}_umap.pdf", dpi=300, bbox_inches='tight')
    plt.close()

In [ ]:
adata.obs.to_csv('embryo_model_integration_scPoli_3round_metadata.csv')